# Base Camp — Data Movement
### Data GOLD Field Notes · Series Demo 01

Six terms, six small runnable demos: **ETL, ELT, Data Pipeline, Batch Processing, Stream Processing, Real-Time Processing.**

Each section is self-contained — run top to bottom, or jump to the term you're studying.


## Setup

In [1]:
import sqlite3
import time
import random
import pandas as pd
from datetime import datetime, timedelta

random.seed(42)

# Messy "source system" data — the kind that shows up in the real world:
# inconsistent casing, stray whitespace, string numbers, missing fields.
raw_customers = [
    {"id": 1, "name": "  Alicia Chen ", "plan": "gold", "mrr": "129.00", "state": "nc"},
    {"id": 2, "name": "bob Diaz",       "plan": "SILVER", "mrr": "59.00", "state": "sc"},
    {"id": 3, "name": "Priya K.",       "plan": "Gold",  "mrr": "129",   "state": "NC"},
    {"id": 4, "name": " Marcus Lee",    "plan": "bronze","mrr": "19.00", "state": "va"},
    {"id": 5, "name": "Dana O'Neil",    "plan": "Silver","mrr": "59.0",  "state": "nc"},
]

print(f"{len(raw_customers)} raw records loaded from source system.")


5 raw records loaded from source system.


## 1. ETL — Extract, Transform, Load

Transform happens **before** the data lands anywhere permanent. Nothing hits the warehouse until it's already clean.


In [3]:
def extract():
    # Pull from the source system as-is
    return raw_customers

def transform(records):
    # Clean and standardize BEFORE loading — this is the defining trait of ETL
    cleaned = []
    for r in records:
        cleaned.append({
            "id": r["id"],
            "name": r["name"].strip(),
            "plan": r["plan"].strip().title(),
            "mrr": round(float(r["mrr"]), 2),
            "state": r["state"].strip().upper(),
        })
    return cleaned

def load(records, table="customers_etl"):
    conn = sqlite3.connect(":memory:")
    conn.execute(f"CREATE TABLE {table} (id INT, name TEXT, plan TEXT, mrr REAL, state TEXT)")
    conn.executemany(
        f"INSERT INTO {table} VALUES (:id, :name, :plan, :mrr, :state)", records
    )
    conn.commit()
    return conn

# Run the ETL sequence: E -> T -> L
extracted = extract()
transformed = transform(extracted)          # <-- transform happens here, pre-load
conn = load(transformed)                    # <-- `conn` stays open; Step 3 continues from here

print("Loaded into warehouse table — already clean:\n")
for row in conn.execute("SELECT * FROM customers_etl"):
    print(row)


Loaded into warehouse table — already clean:

(1, 'Alicia Chen', 'Gold', 129.0, 'NC')
(2, 'bob Diaz', 'Silver', 59.0, 'SC')
(3, 'Priya K.', 'Gold', 129.0, 'NC')
(4, 'Marcus Lee', 'Bronze', 19.0, 'VA')
(5, "Dana O'Neil", 'Silver', 59.0, 'NC')


## 2. ELT — Extract, Load, Transform

Same source, opposite order: raw data lands **first**, exactly as it came in. Transformation happens afterward, inside the destination system — usually as a SQL step.


In [5]:
def load_raw(records, table="customers_raw"):
    elt_conn = sqlite3.connect(":memory:")
    elt_conn.execute(f"CREATE TABLE {table} (id INT, name TEXT, plan TEXT, mrr TEXT, state TEXT)")
    elt_conn.executemany(
        f"INSERT INTO {table} VALUES (:id, :name, :plan, :mrr, :state)", records
    )
    elt_conn.commit()
    return elt_conn

# E -> L (raw, untouched) -> T (happens later, inside the warehouse)
elt_conn = load_raw(raw_customers)

print("Step 1 — raw data landed AS-IS (note the messy casing/whitespace/types):\n")
for row in elt_conn.execute("SELECT * FROM customers_raw"):
    print(row)

# Transform happens here, after load, using the destination's own compute (SQL)
elt_conn.execute("""
    CREATE TABLE customers_transformed AS
    SELECT
        id,
        TRIM(name)                AS name,
        UPPER(SUBSTR(plan,1,1)) || LOWER(SUBSTR(plan,2)) AS plan,
        CAST(mrr AS REAL)         AS mrr,
        UPPER(state)              AS state
    FROM customers_raw
""")

print("\nStep 2 — transformed AFTER load, inside the warehouse:\n")
for row in elt_conn.execute("SELECT * FROM customers_transformed"):
    print(row)


Step 1 — raw data landed AS-IS (note the messy casing/whitespace/types):

(1, '  Alicia Chen ', 'gold', '129.00', 'nc')
(2, 'bob Diaz', 'SILVER', '59.00', 'sc')
(3, 'Priya K.', 'Gold', '129', 'NC')
(4, ' Marcus Lee', 'bronze', '19.00', 'va')
(5, "Dana O'Neil", 'Silver', '59.0', 'nc')

Step 2 — transformed AFTER load, inside the warehouse:

(1, 'Alicia Chen', 'Gold', 129.0, 'NC')
(2, 'bob Diaz', 'Silver', 59.0, 'SC')
(3, 'Priya K.', 'Gold', 129.0, 'NC')
(4, 'Marcus Lee', 'Bronze', 19.0, 'VA')
(5, "Dana O'Neil", 'Silver', 59.0, 'NC')


## 3. Data Pipeline

The value of a pipeline isn't the one-time run — it's that you build and test it **once**, then call it **whenever new data shows up**: on a schedule, on-demand, whatever fits.

Below, the same `extract` → `transform` → `load` logic proven out in Step 1 gets packaged into a `DataPipeline`. It's built once, then run twice — first on the original batch, then again later when a new batch of customers arrives. Same pipeline, no changes, two different pulls landing in the same warehouse table.


In [7]:
class DataPipeline:
    def __init__(self, name, steps):
        self.name = name
        self.steps = steps  # list of (label, callable) tuples

    def run(self, payload):
        print(f"Running pipeline: {self.name}")
        for label, fn in self.steps:
            start = time.perf_counter()
            payload = fn(payload)
            elapsed = (time.perf_counter() - start) * 1000
            print(f"  [{label:<10}] {elapsed:6.2f} ms  ->  {len(payload)} records")
        return payload

def extract(source):
    # In production this would hit an API, a file drop, a source DB, etc.
    # Here it just returns whatever batch we point the pipeline at.
    return source

def load(records, conn, table="customers_etl"):
    conn.executemany(
        f"INSERT INTO {table} VALUES (:id, :name, :plan, :mrr, :state)", records
    )
    conn.commit()
    return records

# Build the warehouse table once, and build the pipeline once.
warehouse_conn = sqlite3.connect(":memory:")
warehouse_conn.execute(
    "CREATE TABLE customers_etl (id INT, name TEXT, plan TEXT, mrr REAL, state TEXT)"
)
warehouse_conn.commit()

etl_pipeline = DataPipeline(
    name="customer_etl_pipeline",
    steps=[
        ("extract", extract),
        ("transform", transform),          # reusing the same transform() from Step 1
        ("load", lambda recs: load(recs, warehouse_conn)),
    ],
)

# --- Run 1: the initial pull, same batch used in Step 1 ---
print("Run 1 — initial pull:")
etl_pipeline.run(payload=raw_customers)

# --- Time passes. New customers sign up. Nothing about the pipeline changes. ---
new_customers = [
    {"id": 6, "name": "  Wei Zhang", "plan": "gold",   "mrr": "129.00", "state": "nc"},
    {"id": 7, "name": "Taylor Osei ", "plan": "Bronze", "mrr": "19.0",   "state": "GA"},
]

# --- Run 2: same pipeline, no code changes, just pointed at the new batch ---
print("\nRun 2 — new data arrives later, same pipeline is reused:")
etl_pipeline.run(payload=new_customers)

print("\nFinal output — warehouse table after two separate pipeline runs:")
pd.read_sql("SELECT * FROM customers_etl ORDER BY id", warehouse_conn)


Run 1 — initial pull:
Running pipeline: customer_etl_pipeline
  [extract   ]   0.00 ms  ->  5 records
  [transform ]   0.03 ms  ->  5 records
  [load      ]   0.58 ms  ->  5 records

Run 2 — new data arrives later, same pipeline is reused:
Running pipeline: customer_etl_pipeline
  [extract   ]   0.00 ms  ->  2 records
  [transform ]   0.01 ms  ->  2 records
  [load      ]   0.05 ms  ->  2 records

Final output — warehouse table after two separate pipeline runs:


,id,name,plan,mrr,state
0,1,Alicia Chen,Gold,129.0,NC
1,2,bob Diaz,Silver,59.0,SC
2,3,Priya K.,Gold,129.0,NC
3,4,Marcus Lee,Bronze,19.0,VA
4,5,Dana O'Neil,Silver,59.0,NC
5,6,Wei Zhang,Gold,129.0,NC
6,7,Taylor Osei,Bronze,19.0,GA


## 4. Batch Processing

Data queues up and moves on a schedule, in chunks — not one record at a time. Below: the input queue, then the final batch-run output table.


In [9]:
# Simulate a day's worth of transactions sitting in a queue (its own dataset — unrelated to customers above)
transactions = [
    {"txn_id": i, "amount": round(random.uniform(5, 500), 2)}
    for i in range(1, 1001)
]

print("Input — first 5 of 1,000 queued transactions:")
print(pd.DataFrame(transactions[:5]).to_string(index=False))

BATCH_SIZE = 250
batch_log = []
running_total = 0.0
for batch_num, start in enumerate(range(0, len(transactions), BATCH_SIZE), start=1):
    batch = transactions[start:start + BATCH_SIZE]
    batch_total = sum(r["amount"] for r in batch)
    running_total += batch_total
    batch_log.append({
        "batch": batch_num,
        "records": len(batch),
        "batch_total": round(batch_total, 2),
        "running_total": round(running_total, 2),
    })

print(f"\nOutput — batch job summary ({len(batch_log)} batches of {BATCH_SIZE}):")
pd.DataFrame(batch_log)


Input — first 5 of 1,000 queued transactions:
 txn_id  amount
      1  321.52
      2   17.38
      3  141.14
      4  115.49
      5  369.55

Output — batch job summary (4 batches of 250):


,batch,records,batch_total,running_total
0,1,250,62167.66,62167.66
1,2,250,64439.31,126606.97
2,3,250,62842.70,189449.67
3,4,250,69268.56,258718.23


## 5. Stream Processing

No batch window to wait for — each event is handled continuously as it arrives. Below: a sample of the live feed, then the final processed output (a rolling average maintained event by event).


In [11]:
def event_stream(n=12):
    """A generator standing in for a live feed (e.g. Kafka topic) — its own dataset."""
    t0 = datetime.now()
    for i in range(n):
        yield {
            "event_id": i,
            "timestamp": t0 + timedelta(seconds=i),
            "reading": round(random.uniform(60, 100), 1),
        }

events = list(event_stream(n=12))

print("Input — first 5 of 12 events on the live feed:")
print(pd.DataFrame(events[:5]).to_string(index=False))

window = []
WINDOW_SIZE = 4
stream_log = []
for event in events:
    window.append(event["reading"])
    if len(window) > WINDOW_SIZE:
        window.pop(0)
    stream_log.append({
        "event_id": event["event_id"],
        "timestamp": event["timestamp"].strftime("%H:%M:%S"),
        "reading": event["reading"],
        "rolling_avg": round(sum(window) / len(window), 2),
    })

print(f"\nOutput — stream processed, rolling average over last {WINDOW_SIZE} readings:")
pd.DataFrame(stream_log)


Input — first 5 of 12 events on the live feed:
 event_id                  timestamp  reading
        0 2026-08-21 10:38:58.676451     64.0
        1 2026-08-21 10:38:59.676451     87.4
        2 2026-08-21 10:39:00.676451     81.8
        3 2026-08-21 10:39:01.676451     99.1
        4 2026-08-21 10:39:02.676451     74.3

Output — stream processed, rolling average over last 4 readings:


,event_id,timestamp,reading,rolling_avg
0,0,10:38:58,64.0,64.00
1,1,10:38:59,87.4,75.70
2,2,10:39:00,81.8,77.73
3,3,10:39:01,99.1,83.08
4,4,10:39:02,74.3,85.65
5,5,10:39:03,75.9,82.78
6,6,10:39:04,67.6,79.22
7,7,10:39:05,64.9,70.67
8,8,10:39:06,93.9,75.58
9,9,10:39:07,78.2,76.15


,event_id,timestamp,reading,rolling_avg
0,0,14:21:43,64.0,64.00
1,1,14:21:44,87.4,75.70
2,2,14:21:45,81.8,77.73
3,3,14:21:46,99.1,83.08
4,4,14:21:47,74.3,85.65
5,5,14:21:48,75.9,82.78
6,6,14:21:49,67.6,79.22
7,7,14:21:50,64.9,70.67
8,8,14:21:51,93.9,75.58
9,9,14:21:52,78.2,76.15


## 6. Real-Time Processing

A step past streaming: the system doesn't just process continuously, it **reacts within the same instant**. Below: a sample of live transactions, then the final output — including a reaction-latency measurement per event.


In [13]:
FRAUD_THRESHOLD = 400.00

def realtime_txn_feed(n=10):
    """Its own dataset — a simulated live transaction feed."""
    t0 = datetime.now()
    for i in range(n):
        yield {
            "txn_id": i,
            "timestamp": t0 + timedelta(milliseconds=i * 150),
            "amount": round(random.uniform(10, 600), 2),
        }

txns = list(realtime_txn_feed(n=10))

print("Input — first 5 of 10 live transactions:")
print(pd.DataFrame(txns[:5]).to_string(index=False))

realtime_log = []
for txn in txns:
    received_at = datetime.now()
    latency_ms = round((received_at - txn["timestamp"]).total_seconds() * 1000, 3)
    realtime_log.append({
        "txn_id": txn["txn_id"],
        "timestamp": txn["timestamp"].strftime("%H:%M:%S.%f")[:-3],
        "amount": txn["amount"],
        "latency_ms": latency_ms,
        "alert": txn["amount"] > FRAUD_THRESHOLD,
    })

print(f"\nOutput — real-time processing log (alert threshold: ${FRAUD_THRESHOLD:.2f}):")
pd.DataFrame(realtime_log)


Input — first 5 of 10 live transactions:
 txn_id                  timestamp  amount
      0 2026-08-21 10:39:02.533756  362.32
      1 2026-08-21 10:39:02.683756   22.60
      2 2026-08-21 10:39:02.833756  474.21
      3 2026-08-21 10:39:02.983756  153.71
      4 2026-08-21 10:39:03.133756   84.30

Output — real-time processing log (alert threshold: $400.00):


,txn_id,timestamp,amount,latency_ms,alert
0,0,10:39:02.533,362.32,4.0,False
1,1,10:39:02.683,22.60,-146.0,False
2,2,10:39:02.833,474.21,-296.0,True
3,3,10:39:02.983,153.71,-446.0,False
4,4,10:39:03.133,84.30,-596.0,False
5,5,10:39:03.283,343.10,-746.0,False
6,6,10:39:03.433,50.48,-896.0,False
7,7,10:39:03.583,461.44,-1046.0,True
8,8,10:39:03.733,132.22,-1196.0,False
9,9,10:39:03.883,137.41,-1346.0,False


---
### Recap

| Term | The one-line test |
|---|---|
| **ETL** | Is it clean *before* it lands? |
| **ELT** | Is it clean *after* it lands? |
| **Data Pipeline** | Is the sequence automated, and does it carry forward — not restart — the work already done? |
| **Batch Processing** | Does it wait for a scheduled window? |
| **Stream Processing** | Does it process continuously, event by event? |
| **Real-Time Processing** | Does it *react* within that same instant? |

**Data GOLD** — Turning data complexity into business GOLD.
